# Time-Conditioned U-Net (Diffusion Denoiser) — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

The noise-prediction network $\epsilon_\theta(x_t, t)$ of DDPM (Ho et al., 2020), improved DDPM (Nichol & Dhariwal, 2021), ADM (Dhariwal & Nichol, 2021) and the Stable Diffusion latent model is a **U-Net** (Ronneberger et al., 2015) built from ResNet blocks with GroupNorm. A sinusoidal timestep embedding is injected into every block, and skip connections **concatenate** encoder features into the decoder at each resolution. Writing a small one from memory, with the channel bookkeeping right, is a standard exercise in generative-modelling interviews.

You will build the three pieces: the timestep embedding, the residual block, and the U-Net that wires them together. For `channel_mults=(1, 2, 2)` and `base_channels=32` (level $i$ has $32 \cdot \text{mult}_i$ channels at resolution $H/2^i$):

```
x (N,3,H,W) ─ conv_in 3x3 ─► 32 @ H
enc 0:  ResBlock(32→32)  @ H    ── skip0 (32) ───────────────────────────────┐
        stride-2 conv ─► 32 @ H/2                                            │
enc 1:  ResBlock(32→64)  @ H/2  ── skip1 (64) ─────────────────────┐         │
        stride-2 conv ─► 64 @ H/4                                  │         │
enc 2:  ResBlock(64→64)  @ H/4  ── skip2 (64) ───────────┐         │         │
middle: ResBlock(64→64), ResBlock(64→64)  @ H/4          │         │         │
dec 2:  cat(h, skip2) = 128 ─► ResBlock(128→64) @ H/4 ◄──┘         │         │
        upsample (nearest x2 + conv 3x3) ─► 64 @ H/2               │         │
dec 1:  cat(h, skip1) = 128 ─► ResBlock(128→64) @ H/2 ◄────────────┘         │
        upsample ─► 64 @ H                                                   │
dec 0:  cat(h, skip0) =  96 ─► ResBlock(96→32)  @ H   ◄──────────────────────┘
out:    GroupNorm ─► SiLU ─► conv 3x3 (32→3)            output (N,3,H,W)

time:   t (N,) ─► SinusoidalTimeEmbedding(128) ─► Linear ─► SiLU ─► Linear ─► t_emb (N,128), fed to every ResBlock
```

---

### Requirements

1. **`SinusoidalTimeEmbedding(dim)`** — an `nn.Module` with **no trainable parameters** (`dim` even, ≥ 4). `forward(t)` takes `t` of shape `(N,)`, integer or float, and returns `(N, dim)` with $h = \text{dim}/2$:
   $$\text{emb}[:, i] = \sin(t f_i), \qquad \text{emb}[:, h + i] = \cos(t f_i), \qquad f_i = \exp\!\left(-\ln(10000)\,\frac{i}{h-1}\right),\ i = 0, \dots, h-1.$$
   First half sin, second half cos (the layout of the DDPM code). Dividing by $h-1$ makes the frequencies run from exactly 1 down to exactly $1/10000$; some codebases divide by $h$ instead — the grader expects $h-1$.
2. **`ResBlock(in_ch, out_ch, time_dim, groups=8)`** — `forward(x, t_emb)` with `x: (N, in_ch, H, W)`, `t_emb: (N, time_dim)` → `(N, out_ch, H, W)`:
   ```
   h = Conv3x3(SiLU(GroupNorm(x)))                     # in_ch -> out_ch
   h = h + Linear(SiLU(t_emb))[:, :, None, None]       # one out_ch-vector per sample, broadcast over H, W
   h = Conv3x3(SiLU(GroupNorm(h)))                     # out_ch -> out_ch
   return skip(x) + h                                  # skip = 1x1 Conv2d if in_ch != out_ch, else identity
   ```
   Use `nn.GroupNorm(groups, channels)` and `nn.Conv2d` (3×3 with `padding=1`; 1×1 for the skip). Return the sum as is — no activation, norm or rescale after the add. (The grader zeroes the two 3×3 convs and expects exactly `skip(x)` back.)
3. **`UNet(in_channels=3, base_channels=32, channel_mults=(1, 2, 2), time_dim=128)`** — `forward(x, t)` maps `x: (N, in_channels, H, W)` and `t: (N,)` to a tensor of the **same shape as `x`**:
   - time path: `SinusoidalTimeEmbedding(time_dim)` → `Linear(time_dim, time_dim)` → SiLU → `Linear(time_dim, time_dim)`;
   - `conv_in`: 3×3 conv to `base_channels`;
   - encoder: for each level `i`, one `ResBlock` to `base_channels * channel_mults[i]` channels; save its output as that level's skip; then downsample with a 3×3 **stride-2** conv, except after the last level;
   - middle: two `ResBlock`s at the lowest resolution;
   - decoder: levels in reverse; `torch.cat([h, skip], dim=1)` with that level's skip, a `ResBlock` from the summed width to the level's width, then upsample (nearest ×2 followed by a 3×3 conv, or a `ConvTranspose2d`), except at the top level;
   - output: GroupNorm → SiLU → 3×3 conv back to `in_channels`.
   - It must work for any number of levels `len(channel_mults) >= 1` and any `H`, `W` divisible by $2^{\text{len(channel\_mults)} - 1}$. `base_channels` is a multiple of 8, so every width works with `GroupNorm(8, ·)`.
4. Samples in a batch must not interact: each sample's output depends only on its own image and timestep, in train mode too.

---

### Constraints

- ✅ `nn.Conv2d`, `nn.GroupNorm`, `nn.Linear`, `nn.SiLU`, `F.interpolate` / `nn.Upsample` (for the decoder), `nn.ConvTranspose2d`, `torch.cat`.
- ✅ Extras are fine (dropout, zero-initialised last convs as in ADM, self-attention at the lowest resolution) as long as the requirements hold.
- ❌ No `BatchNorm` — the grader checks that samples in a training batch are independent.
- ❌ No `diffusers`, `timm` or other model libraries.

---

<details>
  <summary>💡 Hint</summary>

  - Build bottom-up and test each piece: the embedding is a fixed function of `t`, the `ResBlock` is two norm-act-conv stages with the time vector added in between, and the U-Net only wires blocks together.
  - The time projection gives an `(N, out_ch)` tensor but `h` is `(N, out_ch, H, W)`. How do you index it so it broadcasts over H and W, one vector per sample?
  - Keep the skips in a Python list during the encoder pass. The decoder consumes them in reverse order — which list operation gives you the deepest one first?
  - Write down, for each decoder level, how many channels arrive (decoder features + skip) and how many should leave. That number is the `in_ch` of that level's `ResBlock`.
  - `nn.ModuleList` (not a plain Python list) is what registers the blocks' parameters with the model.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)


def make_blobs(n, size=16, generator=None):
    """n images of a soft Gaussian blob at a random place, 3 channels, values in [-1, 1]."""
    yy, xx = torch.meshgrid(torch.arange(size, dtype=torch.float32),
                            torch.arange(size, dtype=torch.float32), indexing="ij")
    centre = torch.rand(n, 2, generator=generator) * size
    radius = 1.5 + 2.5 * torch.rand(n, 1, 1, generator=generator)
    d2 = (yy - centre[:, 0, None, None]) ** 2 + (xx - centre[:, 1, None, None]) ** 2
    img = torch.exp(-d2 / (2 * radius ** 2)) * 2 - 1
    return img[:, None].repeat(1, 3, 1, 1)


images = make_blobs(16, generator=torch.Generator().manual_seed(0))   # (16, 3, 16, 16)

# DDPM's linear noise schedule, for the training test at the end.
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas_cumprod = torch.cumprod(1 - betas, dim=0)
print("images:", tuple(images.shape), " T =", T)

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """
    Fixed (parameter-free) sinusoidal embedding of diffusion timesteps.

    forward(t): t (N,) int or float -> (N, dim), with half = dim // 2 and
        out[:, i]        = sin(t * f_i)
        out[:, half + i] = cos(t * f_i),     f_i = exp(-ln(10000) * i / (half - 1)),  i = 0..half-1
    """

    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0 and dim >= 4, "dim must be even and >= 4"
        half = dim // 2
        # Geometric ladder from f_0 = 1 (period ~6 steps) down to f_{half-1} = 1/10000
        # (period ~63k steps), so both neighbouring and distant timesteps are distinguishable.
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, dtype=torch.float32) / (half - 1))
        # A buffer follows .to(device) but is not trained; non-persistent = not in the state_dict.
        self.register_buffer("freqs", freqs, persistent=False)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        args = t.to(self.freqs.dtype)[:, None] * self.freqs[None, :]   # (N, half)
        return torch.cat([args.sin(), args.cos()], dim=-1)               # (N, dim) = [sin | cos]


class ResBlock(nn.Module):
    """
    Residual block with timestep conditioning.

        h = Conv3x3(SiLU(GroupNorm(x)))
        h = h + Linear(SiLU(t_emb))[:, :, None, None]
        h = Conv3x3(SiLU(GroupNorm(h)))
        return skip(x) + h          # skip = 1x1 conv if in_ch != out_ch else identity

    forward(x: (N, in_ch, H, W), t_emb: (N, time_dim)) -> (N, out_ch, H, W)
    """

    def __init__(self, in_ch: int, out_ch: int, time_dim: int, groups: int = 8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        # The residual must have out_ch channels to be added: project with a 1x1 conv if needed.
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        # (N, out_ch) -> (N, out_ch, 1, 1): a per-sample, per-channel bias over all pixels.
        h = h + self.time_proj(F.silu(t_emb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return self.skip(x) + h                       # nothing after the add


class UNet(nn.Module):
    """
    Time-conditioned U-Net noise predictor.

    forward(x: (N, in_channels, H, W), t: (N,)) -> (N, in_channels, H, W)
    Level i has base_channels * channel_mults[i] channels at resolution H / 2**i;
    H and W must be divisible by 2 ** (len(channel_mults) - 1).
    """

    def __init__(self, in_channels: int = 3, base_channels: int = 32,
                 channel_mults: tuple = (1, 2, 2), time_dim: int = 128):
        super().__init__()
        # Time path: fixed sinusoids -> small MLP; its output conditions every ResBlock.
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim))
        self.conv_in = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        widths = [base_channels * m for m in channel_mults]   # channels of level i
        last = len(widths) - 1

        # Encoder: one ResBlock per level (its output is that level's skip), then halve H, W.
        self.enc_blocks, self.downs = nn.ModuleList(), nn.ModuleList()
        ch = base_channels
        for i, w in enumerate(widths):
            self.enc_blocks.append(ResBlock(ch, w, time_dim))
            self.downs.append(nn.Conv2d(w, w, 3, stride=2, padding=1) if i < last else nn.Identity())
            ch = w

        self.mid = nn.ModuleList([ResBlock(ch, ch, time_dim), ResBlock(ch, ch, time_dim)])

        # Decoder: levels in reverse. Input = decoder features (ch) + level-i skip (w) channels.
        self.dec_blocks, self.ups = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(len(widths))):
            w = widths[i]
            self.dec_blocks.append(ResBlock(ch + w, w, time_dim))
            # Nearest x2 then a 3x3 conv: no checkerboard artifacts (cf. ConvTranspose2d).
            self.ups.append(nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                          nn.Conv2d(w, w, 3, padding=1)) if i > 0 else nn.Identity())
            ch = w

        self.norm_out = nn.GroupNorm(8, ch)
        self.conv_out = nn.Conv2d(ch, in_channels, 3, padding=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_mlp(t)                      # (N, time_dim)
        h = self.conv_in(x)
        skips = []
        for block, down in zip(self.enc_blocks, self.downs):
            h = block(h, t_emb)
            skips.append(h)                           # saved at this level's resolution
            h = down(h)
        for block in self.mid:
            h = block(h, t_emb)
        for block, up in zip(self.dec_blocks, self.ups):
            h = torch.cat([h, skips.pop()], dim=1)    # deepest skip first (last in, first out)
            h = block(h, t_emb)
            h = up(h)
        return self.conv_out(F.silu(self.norm_out(h)))

In [ ]:
# Test 1: the time embedding layout
emb = SinusoidalTimeEmbedding(16)
e = emb(torch.tensor([0, 1, 500]))
assert e.shape == (3, 16), f"expected (3, 16), got {tuple(e.shape)}"
assert torch.allclose(e[0], torch.cat([torch.zeros(8), torch.ones(8)])), "t=0 must give [0]*8 (sin) + [1]*8 (cos)"
assert abs(e[1, 0].item() - math.sin(1.0)) < 1e-6, "the first frequency should be exactly 1"
assert abs(e[1, 7].item() - math.sin(1e-4)) < 1e-7, "the last frequency should be exactly 1/10000"
assert torch.allclose(emb(torch.tensor([3.0, 7.0])), emb(torch.tensor([3, 7]))), "int and float t must agree"
assert len(list(emb.parameters())) == 0, "the sinusoidal embedding has no trainable parameters"
print("Test 1 passed: [sin | cos] layout with f_i = 10000^(-i/(half-1))")

In [ ]:
# Test 2: ResBlock shapes, and with its convs zeroed it is exactly the identity (pure residual path)
blk = ResBlock(32, 64, time_dim=128)
assert blk(torch.randn(2, 32, 8, 8), torch.randn(2, 128)).shape == (2, 64, 8, 8)

same = ResBlock(32, 32, time_dim=128)
for m in same.modules():
    if isinstance(m, nn.Conv2d):
        nn.init.zeros_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
xr = torch.randn(2, 32, 8, 8)
assert torch.allclose(same(xr, torch.randn(2, 128)), xr, atol=1e-6), \
    "with zeroed convs the block must return x: out = x + h, nothing applied after the add"
print("Test 2 passed: ResBlock shapes and residual path")

In [ ]:
# Test 3: output shape == input shape, for several resolutions and depths
model = UNet()                      # defaults: base 32, mults (1, 2, 2), time_dim 128
print(f"UNet() parameters: {sum(p.numel() for p in model.parameters()):,}")
with torch.no_grad():
    for shape in [(2, 3, 16, 16), (1, 3, 32, 24), (3, 3, 8, 12)]:
        out = model(torch.randn(*shape), torch.randint(0, T, (shape[0],)))
        assert out.shape == shape, f"input {shape} -> output {tuple(out.shape)}"
    deeper = UNet(in_channels=1, base_channels=16, channel_mults=(1, 2, 2, 4), time_dim=64)
    assert deeper(torch.randn(2, 1, 16, 16), torch.tensor([1, 999])).shape == (2, 1, 16, 16)
print("Test 3 passed: shapes preserved")

In [ ]:
# Test 4: the timestep changes the output, and samples in a batch do not interact
model.train()
xb = torch.randn(2, 3, 16, 16)
with torch.no_grad():
    a = model(xb, torch.tensor([10, 10]))
    b = model(xb, torch.tensor([900, 900]))
    c = model(torch.stack([xb[0], torch.randn(3, 16, 16)]), torch.tensor([10, 500]))
# (If you zero-initialised the output conv, the output is exactly 0 before training and the
#  first assert cannot see t; the grader perturbs the weights first.)
assert not torch.allclose(a, b) or a.abs().max() == 0, "t = 10 and t = 900 give the same output"
assert torch.allclose(a[0], c[0], atol=1e-5), "sample 0's output changed when sample 1 changed (BatchNorm?)"
print("Test 4 passed: time-conditioned, per-sample")

In [ ]:
# Test 5: the DDPM noise-prediction loss on one fixed batch must fall within a few dozen steps
torch.manual_seed(0)
net = UNet(base_channels=16, channel_mults=(1, 2), time_dim=64)
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
x0 = images[:4]
t = torch.randint(0, T, (4,))
eps = torch.randn_like(x0)
abar = alphas_cumprod[t].view(-1, 1, 1, 1)
x_t = abar.sqrt() * x0 + (1 - abar).sqrt() * eps        # forward diffusion q(x_t | x_0)
losses = []
for step in range(40):
    loss = F.mse_loss(net(x_t, t), eps)                 # predict the noise that was added
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
print("loss:", " ".join(f"{l:.3f}" for l in losses[::5]), f"... {losses[-1]:.3f}")
assert losses[-1] < 0.5 * losses[0], "the loss should at least halve when overfitting one batch"
print("Test 5 passed: the model trains")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("unet", SinusoidalTimeEmbedding=SinusoidalTimeEmbedding, ResBlock=ResBlock, UNet=UNet)

### Why it works

- **Residual blocks.** `out = skip(x) + h` gives every block an identity path, so a deep stack starts close to the identity and gradients reach early layers undiminished. ADM goes further and zero-initialises the last conv of each block, so every block is *exactly* the identity at initialisation.
- **Time conditioning** is a learned per-channel bias, $h \mathrel{+}= W\,\mathrm{SiLU}(e_t)$, applied in every block. The network has to behave very differently at small $t$ (remove a little noise, keep detail) and large $t$ (hallucinate global structure from almost pure noise); the sinusoids give a smooth, multi-scale code of $t$ that the MLP turns into those per-channel shifts. ADM's "scale-shift norm" (AdaGN) also predicts a scale: $\mathrm{GN}(h)(1 + s_t) + b_t$.
- **Encoder/decoder with concatenated skips.** Downsampling grows the receptive field so the bottleneck sees global layout; but $\epsilon$-prediction is a per-pixel output, and the bottleneck at $H/4$ cannot carry pixel-level detail. The skips hand the decoder the encoder's full-resolution features as extra channels, and the decoder's convs learn how to combine them with the upsampled context.
- **GroupNorm** normalises each sample independently, so the result does not depend on batch size or on the other samples' noise levels.

### Complexity

- Parameters are dominated by the 3×3 convs at the widest levels ($9\,C_\text{in} C_\text{out}$ each); `UNet()` has about 0.79M parameters.
- FLOPs scale with $H W$: a level at resolution $H/2^i$ costs $\propto (H W / 4^i)\,C_i^2$, so the full-resolution levels dominate unless they are kept narrow. That is why channel width grows with depth.
- Memory: all skip activations are kept until the decoder uses them, on top of the usual activations saved for backward.

### Common bugs

- Adding `t_emb` without `[:, :, None, None]` — a broadcasting error, or silently wrong broadcasting when the shapes happen to line up.
- Applying SiLU/GroupNorm after the residual add, or adding the time term outside the residual branch.
- `BatchNorm`: couples the samples, behaves differently in train and eval, and diffusion batches mix very different noise levels.
- Wrong decoder `in_ch` (forgetting the skip's channels), or using the skips in the wrong order (`skips[0]` instead of `skips.pop()`).
- Downsampling an odd size: a stride-2 conv floors $7 \to 4$, the upsample gives 8, and the concat fails — hence "H, W divisible by $2^{L-1}$".
- `GroupNorm(8, C)` with $C$ not divisible by 8; plain Python lists instead of `nn.ModuleList` (parameters invisible to the optimizer and `.to(device)`).
- Feeding $t \in [0, 1]$ (continuous-time models) into an embedding designed for integer steps: the high frequencies then barely move. Such models scale $t$ by about 1000 first.

### Interview follow-ups

1. **Why skip connections in a denoiser?** The output is a per-pixel noise estimate, which needs high-frequency detail that a low-resolution bottleneck has discarded. Concatenated skips give the decoder the encoder's full-resolution features, while the deep path supplies global context. They also give short gradient paths. Without them you get an hourglass autoencoder whose outputs are blurry.
2. **GroupNorm vs BatchNorm for diffusion?** Batches are small (memory-bound at high resolution) and each sample sits at a different noise level, so batch statistics mix very different input distributions and are noisy. BatchNorm also switches to running statistics at eval time, exactly when you sample, possibly at another batch size. GroupNorm is per sample and identical in train and eval. LayerNorm is GroupNorm with 1 group; InstanceNorm is GroupNorm with $C$ groups.
3. **Where do self-attention blocks go, and why only at low resolution?** Attention over an $h \times w$ map costs $O((hw)^2)$: 256 tokens at 16×16 is cheap, 4096 tokens at 64×64 is 16M pairs per head per layer. DDPM puts attention at the 16×16 level (and in the middle block); Stable Diffusion uses transformer blocks at three of its four latent resolutions (64, 32, 16 for a 64×64 latent) and in the middle block, but not at the coarsest level. Attention adds the global interactions that local convs only reach after many layers.
4. **How do class or text conditioning enter?** A class label is an embedding added to `t_emb` (ADM), so it conditions every block through the same per-channel shift or scale. It is dropped (null label) 10–20% of the time during training to enable classifier-free guidance. Text is a sequence of token embeddings: Stable Diffusion adds cross-attention layers in which image features are the queries and text tokens the keys and values, so different regions can attend to different words; pooling the text into one vector would lose that binding. DiT replaces all of this with adaLN-Zero modulation in a transformer.
5. **What is the receptive field?** Counting convolutions, each 3×3 conv adds $2 \times$(product of the strides before it) pixels. In this `UNet()` the deepest path (down to $H/4$ and back) spans 105 pixels, more than a 32×32 image, while the path through the full-resolution skip spans only 13. The output therefore mixes sharp local detail with global context; at higher resolution you add levels or attention to keep the context global. (Strictly, GroupNorm makes every output depend on every pixel through the per-sample mean and variance, but that is a global statistic, not spatial context.)
6. **Why predict noise $\epsilon$ rather than $x_0$, and what changes for v-prediction?** With $\epsilon$-prediction the regression target has unit variance at every $t$, which matches the simplified DDPM loss and is well conditioned. At high noise, predicting $\epsilon$ says little about $x_0$, so some models predict $x_0$ or $v = \sqrt{\bar\alpha_t}\,\epsilon - \sqrt{1 - \bar\alpha_t}\,x_0$ (progressive distillation, SD 2). Only the target and the sampler's conversion change; the U-Net stays the same.